<h1 style="color:DodgerBlue">Индивидальный проект</h1>

<h2 style="color:DodgerBlue">Название проекта:</h2>

Зоопарк

----

### Вариант задания 

Система «Зоопарк»: животные, склад корма и смотритель.


<h2 style="color:DodgerBlue">Описание проекта:</h2>

----

Продолжение проекта «Зоопарк». Базовый класс `Animal` (4 атрибута и 4 метода) и производные `Lion`, `Monkey`, `Penguin`. Явная реализация интерфейсов: `IFeedable.Feed` в `Animal`, `IWalker.Move`/`ISwimmer.Move` в `Penguin`. Управление зависимостями: `Zookeeper` получает `ILogger` и `IFoodStorage` через конструктор.

#### Дополнительное задание
Добавьте к сущестующим классам (базовыму и производным 3-4 атрибута и метода) создайте явную реализации интерфейса и управление зависимостями 


<h2 style="color:DodgerBlue">Реализация:</h2>

----

In [ ]:
using System;
using System.Collections.Generic;

// ===== Интерфейсы =====
public interface ILogger { void Log(string message); }

public interface IFoodStorage
{
    bool Take(string food, int grams);
    int Stock(string food);
}

public interface IFeedable
{
    string Name { get; }
    void Feed(string food, int grams);
}

public interface IWalker  { void Move(); }
public interface ISwimmer { void Move(); }

// ===== Реализации зависимостей =====
public class ConsoleLogger : ILogger
{
    public void Log(string message) => Console.WriteLine($"[ЗООПАРК] {message}");
}

public class FoodStorage : IFoodStorage
{
    private Dictionary<string, int> _stock = new Dictionary<string, int>();

    public void Add(string food, int grams)
    {
        _stock[food] = Stock(food) + grams;
    }

    public int Stock(string food)
    {
        return _stock.ContainsKey(food) ? _stock[food] : 0;
    }

    public bool Take(string food, int grams)
    {
        if (Stock(food) < grams) return false;
        _stock[food] -= grams;
        return true;
    }
}

// ===== Базовый класс: 4 атрибута, 4 метода =====
public abstract class Animal : IFeedable
{
    private string _name;
    private int _age;
    private double _weight;
    private int _energy = 50;

    public string Name   { get { return _name; }   set { _name = value; } }
    public int Age       { get { return _age; }    set { _age = value < 0 ? 0 : value; } }
    public double Weight { get { return _weight; } set { _weight = value; } }
    public int Energy    { get { return _energy; } protected set { _energy = Math.Max(0, Math.Min(100, value)); } }

    protected Animal(string name, int age, double weight)
    {
        Name = name;
        Age = age;
        Weight = weight;
    }

    public abstract void MakeSound();

    public void Eat(string food, int grams)
    {
        Energy += grams / 20;
        Weight += grams / 1000.0;
        Console.WriteLine($"{Name} съел {food} ({grams} г). Энергия: {Energy}");
    }

    public void Sleep()
    {
        Energy += 30;
        Console.WriteLine($"{Name} поспал. Энергия: {Energy}");
    }

    public virtual void DisplayInfo() =>
        Console.WriteLine($"{GetType().Name} {Name}, {Age} лет, {Weight:F1} кг, энергия {Energy}");

    // Явная реализация интерфейса: Feed доступен только через IFeedable
    void IFeedable.Feed(string food, int grams) => Eat(food, grams);
}

// ===== Производные классы =====
public class Lion : Animal, IWalker
{
    public int PrideSize { get; set; } = 5;
    public int RoarPower { get; set; } = 80;
    public double ManeLength { get; set; } = 0.4;

    public Lion(string name, int age, double weight) : base(name, age, weight) { }

    public override void MakeSound() => Console.WriteLine($"{Name}: Рррр!");
    public void Roar() => Console.WriteLine($"{Name} рычит с силой {RoarPower}");
    public void Hunt() { Energy -= 25; Console.WriteLine($"{Name} охотится. Энергия: {Energy}"); }
    public void LeadPride() => Console.WriteLine($"{Name} ведёт прайд из {PrideSize} особей");

    void IWalker.Move() => Console.WriteLine($"{Name} идёт по саванне");
}

public class Monkey : Animal
{
    public int Intelligence { get; set; } = 7;
    public double TailLength { get; set; } = 0.6;
    public int Bananas { get; set; }

    public Monkey(string name, int age, double weight) : base(name, age, weight) { }

    public override void MakeSound() => Console.WriteLine($"{Name}: Уу-уу!");
    public void Climb() => Console.WriteLine($"{Name} лезет на дерево");
    public void CollectBanana() { Bananas++; Console.WriteLine($"{Name} собрал бананов: {Bananas}"); }
    public void UseTool() => Console.WriteLine($"{Name} использует инструмент");
}

// Два интерфейса с одинаковым Move() -> только явная реализация
public class Penguin : Animal, IWalker, ISwimmer
{
    public double DiveDepth { get; set; } = 100;
    public int FeatherCount { get; set; } = 30000;
    public string Colony { get; set; } = "Антарктическая";

    public Penguin(string name, int age, double weight) : base(name, age, weight) { }

    public override void MakeSound() => Console.WriteLine($"{Name}: Кррря!");
    public void Slide() => Console.WriteLine($"{Name} скользит на животе");
    public void Dive() { Energy -= 10; Console.WriteLine($"{Name} ныряет на {DiveDepth} м. Энергия: {Energy}"); }

    void IWalker.Move()  => Console.WriteLine($"{Name} переваливается по льду");
    void ISwimmer.Move() => Console.WriteLine($"{Name} быстро плывёт");
}

// ===== Управление зависимостями (внедрение через конструктор) =====
public class Zookeeper
{
    private readonly ILogger _logger;
    private readonly IFoodStorage _storage;

    public string Name { get; set; }

    public Zookeeper(string name, ILogger logger, IFoodStorage storage)
    {
        Name = name;
        _logger = logger;
        _storage = storage;
    }

    public void FeedAnimal(IFeedable animal, string food, int grams)
    {
        if (_storage.Take(food, grams))
        {
            _logger.Log($"{Name} кормит {animal.Name}: {food}, {grams} г");
            animal.Feed(food, grams);
        }
        else
        {
            _logger.Log($"Не хватает корма «{food}» для {animal.Name} (на складе {_storage.Stock(food)} г)");
        }
    }
}

// ===== Корень композиции =====
ILogger logger = new ConsoleLogger();
FoodStorage storage = new FoodStorage();
storage.Add("мясо", 3000);
storage.Add("рыба", 1000);
storage.Add("бананы", 500);

Zookeeper keeper = new Zookeeper("Иван", logger, storage);

Lion lion = new Lion("Симба", 6, 190);
Monkey monkey = new Monkey("Чита", 4, 25);
Penguin penguin = new Penguin("Пингви", 3, 15);

keeper.FeedAnimal(lion, "мясо", 2000);
keeper.FeedAnimal(penguin, "рыба", 600);
keeper.FeedAnimal(monkey, "бананы", 300);
keeper.FeedAnimal(lion, "мясо", 2000);   // не хватит корма

lion.Hunt();
monkey.Climb();

// lion.Feed("мясо", 100);      // ошибка компиляции: явная реализация скрыта
IWalker walker = penguin;
walker.Move();
ISwimmer swimmer = penguin;
swimmer.Move();
((IWalker)lion).Move();

List<Animal> zoo = new List<Animal> { lion, monkey, penguin };
foreach (Animal a in zoo) a.DisplayInfo();